In [7]:
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

# Use relative paths from notebook location
PROJECT_ROOT = Path("../..").resolve()
npy_folder_path = PROJECT_ROOT / "data" / "processed" / "minimal_exploration"
print(f"Loading features from: {npy_folder_path}")

Loading features from: D:\dev\karst-terrain-classification\data\processed\minimal_exploration


In [8]:
# Diagnostic: Check for extreme values in raw features
print("="*60)
print("RAW FEATURE VALUE RANGES")
print("="*60)

for name in ['traditional', 'ecc50', 'ecc200', 'betti100', 'betti200', 
             'landscape', 'persimage', 'ph_directional']:
    filepath = npy_folder_path / f'testbed_features_{name}.npy'
    X = np.load(filepath)
    
    min_val = X.min()
    max_val = X.max()
    max_abs = np.abs(X).max()
    has_extreme = np.any(np.abs(X) > 1e100)
    has_large = np.any(np.abs(X) > 1e10)
    
    print(f"{name:15s}: min={min_val:>12.2e}, max={max_val:>12.2e}, "
          f"max_abs={max_abs:>12.2e}, extreme={has_extreme}, large={has_large}")

print("="*60)

RAW FEATURE VALUE RANGES
traditional    : min=   -8.35e+00, max=    2.69e+01, max_abs=    2.69e+01, extreme=False, large=False
ecc50          : min=   -3.29e+01, max=    2.48e+01, max_abs=    3.29e+01, extreme=False, large=False
ecc200         : min=   -3.96e+01, max=    3.10e+01, max_abs=    3.96e+01, extreme=False, large=False
betti100       : min=    0.00e+00, max=    6.20e+01, max_abs=    6.20e+01, extreme=False, large=False
betti200       : min=    0.00e+00, max=    5.90e+01, max_abs=    5.90e+01, extreme=False, large=False
landscape      : min=    0.00e+00, max=    5.62e+01, max_abs=    5.62e+01, extreme=False, large=False
persimage      : min=    0.00e+00, max=    6.74e+01, max_abs=    6.74e+01, extreme=False, large=False
ph_directional : min=    0.00e+00, max=    1.70e+01, max_abs=    1.70e+01, extreme=False, large=False


In [9]:
# Configuration for feature sets
FEATURE_NAMES = ['traditional', 'ecc50', 'ecc200', 'betti100', 'betti200', 
                 'landscape', 'persimage', 'ph_directional']

# Load all feature matrices
print(f"Loading features from: {npy_folder_path}\n")

features = {}
for name in FEATURE_NAMES:
    filepath = npy_folder_path / f'testbed_features_{name}.npy'
    features[name] = np.load(filepath)
    print(f"Loaded {name:15s}: {features[name].shape}")

# Load labels and timing
y = np.load(npy_folder_path / 'testbed_labels.npy')
timing = np.load(npy_folder_path / 'testbed_timing.npy', allow_pickle=True).item()

# Filter out rare labels (alluvial_fans=2, alluvial_terraces=3)
# Keep indices: 0, 1, 4, 5, 6 (artificial_fill, alluvium, colluvium, colluvial_accumulations, residuum)
KEEP_LABEL_INDICES = [0, 1, 4, 5, 6]
y = y[:, KEEP_LABEL_INDICES]
print(f"\n✓ Filtered labels: Removed alluvial_fans and alluvial_terraces (had 0 and 4 examples)")
print(f"✓ Data: {len(y)} tiles, {y.shape[1]} labels (multi-label)")


# Data quality check
print(f"\n{'='*60}")
print("DATA QUALITY CHECK")
print(f"{'='*60}")

def check_data_quality(X, name):
    """Check for inf/NaN values in feature matrix."""
    n_inf = np.isinf(X).sum()
    n_nan = np.isnan(X).sum()
    n_total = X.size
    
    if n_inf > 0 or n_nan > 0:
        print(f"⚠ {name:15s}: {n_inf} inf ({100*n_inf/n_total:.2f}%), "
              f"{n_nan} NaN ({100*n_nan/n_total:.2f}%)")
        return False
    print(f"✓ {name:15s}: Clean (no inf/NaN)")
    return True

all_clean = all(check_data_quality(features[name], name.title()) 
                for name in FEATURE_NAMES)

# Clean data if needed
if not all_clean:
    print(f"\n{'='*60}")
    print("CLEANING DATA")
    print(f"{'='*60}")
    
    def clean_features(X, name):
        """
        Replace inf with large finite values and NaN with 0.
        Uses column-wise statistics for more intelligent replacement.
        """
        X_clean = X.copy()
        
        for col_idx in range(X_clean.shape[1]):
            col = X_clean[:, col_idx]
            finite_mask = np.isfinite(col)
            
            if finite_mask.any():
                # Use column statistics for replacement
                col_mean = np.mean(col[finite_mask])
                col_std = np.std(col[finite_mask])
                col_max = np.max(col[finite_mask])
                col_min = np.min(col[finite_mask])
                
                # Replace inf with 3 std devs beyond the max/min
                pos_replacement = min(col_max + 3 * col_std, np.finfo(np.float32).max / 100)
                neg_replacement = max(col_min - 3 * col_std, np.finfo(np.float32).min / 100)
                
                X_clean[np.isposinf(col), col_idx] = pos_replacement
                X_clean[np.isneginf(col), col_idx] = neg_replacement
                X_clean[np.isnan(col), col_idx] = col_mean
            else:
                # If entire column is non-finite, replace with 0
                X_clean[:, col_idx] = 0.0
        
        return X_clean
    
    features = {name: clean_features(X, name.title()) 
                for name, X in features.items()}
    print("✓ All feature matrices cleaned")
else:
    print("\n✓ All datasets are clean - no preprocessing needed")

# Normalize features
print(f"\n{'='*60}")
print("FEATURE NORMALIZATION")
print(f"{'='*60}")

def safe_standardize(X, name):
    """
    Robustly Standardize features (mean=0, std=1).
    
    Handles:
    1. NaN values (replaces with 0 before scaling)
    2. Inf values (replaces with 0 before scaling)
    3. Constant columns (std=0), which are set to 0.
    """
    import warnings
    
    X_scaled = np.zeros_like(X, dtype=np.float32)
    n_constant = 0
    n_invalid = 0
    
    # Process column-by-column
    for col_idx in range(X.shape[1]):
        col = X[:, col_idx].astype(np.float64)
        
        # 1. Find and count invalid values (NaN or Inf)
        invalid_mask = ~np.isfinite(col)
        if invalid_mask.any():
            n_invalid += invalid_mask.sum()
            col[invalid_mask] = 0.0 # Replace with 0
            
        # 2. Suppress warnings for std calculation
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=RuntimeWarning)
            col_std = np.std(col)
            
            # 3. Check for constant column (std=0 or std=NaN)
            if col_std == 0 or not np.isfinite(col_std):
                X_scaled[:, col_idx] = 0.0 # Set entire column to 0
                n_constant += 1
                continue
            
            # 4. Standardize the column
            col_mean = np.mean(col)
            X_scaled[:, col_idx] = (col - col_mean) / col_std
    
    # Report what we did
    if n_constant > 0 or n_invalid > 0:
        parts = []
        if n_constant > 0:
            parts.append(f"{n_constant} constant features")
        if n_invalid > 0:
            parts.append(f"{n_invalid} inf/NaN values zeroed")
        print(f"  {name:15s}: {', '.join(parts)}")
    
    return X_scaled.astype(np.float32)

features = {name: safe_standardize(X, name.title()) 
            for name, X in features.items()}

print("✓ All features normalized (mean=0, std=1)")
print("  This prevents overflow in sklearn and improves performance")

Loading features from: D:\dev\karst-terrain-classification\data\processed\minimal_exploration

Loaded traditional    : (100, 150)
Loaded ecc50          : (100, 50)
Loaded ecc200         : (100, 200)
Loaded betti100       : (100, 200)
Loaded betti200       : (100, 400)
Loaded landscape      : (100, 1100)
Loaded persimage      : (100, 5000)
Loaded ph_directional : (100, 800)

✓ Filtered labels: Removed alluvial_fans and alluvial_terraces (had 0 and 4 examples)
✓ Data: 100 tiles, 5 labels (multi-label)

DATA QUALITY CHECK
✓ Traditional    : Clean (no inf/NaN)
✓ Ecc50          : Clean (no inf/NaN)
✓ Ecc200         : Clean (no inf/NaN)
✓ Betti100       : Clean (no inf/NaN)
✓ Betti200       : Clean (no inf/NaN)
✓ Landscape      : Clean (no inf/NaN)
✓ Persimage      : Clean (no inf/NaN)
✓ Ph_Directional : Clean (no inf/NaN)

✓ All datasets are clean - no preprocessing needed

FEATURE NORMALIZATION
  Traditional    : 75 constant features
  Ecc50          : 1 constant features
  Ecc200         

In [10]:
# Label distribution analysis
LABEL_NAMES = ['artificial_fill', 'alluvium', 'colluvium', 
               'colluvial_accumulations', 'residuum']

print("="*60)
print("LABEL DISTRIBUTION ANALYSIS")
print("="*60)

label_counts = y.sum(axis=0)
label_percentages = (label_counts / len(y)) * 100

for i, (name, count, pct) in enumerate(zip(LABEL_NAMES, label_counts, label_percentages)):
    print(f"{i}. {name:<25s}: {int(count):>3d} tiles ({pct:>5.1f}%)")

print(f"\nTotal tiles: {len(y)}")
print(f"Average labels per tile: {y.sum(axis=1).mean():.2f}")

# Check for very rare labels
RARE_THRESHOLD = 5
rare_labels = [(i, name, int(count)) 
               for i, (name, count) in enumerate(zip(LABEL_NAMES, label_counts)) 
               if count < RARE_THRESHOLD]

if rare_labels:
    print(f"\n⚠ WARNING: Found {len(rare_labels)} rare label(s) with < {RARE_THRESHOLD} examples:")
    for idx, name, count in rare_labels:
        print(f"  - Label {idx} ({name}): {count} examples")
    print("\nThese labels may cause issues with cross-validation!")
    print("Options:")
    print("  1. Remove rare labels from evaluation")
    print("  2. Use stratified multi-label splits (may still fail)")
    print("  3. Increase dataset size")
else:
    print(f"\n✓ All labels have at least {RARE_THRESHOLD} examples")

LABEL DISTRIBUTION ANALYSIS
0. artificial_fill          :  34 tiles ( 34.0%)
1. alluvium                 :  84 tiles ( 84.0%)
2. colluvium                :  30 tiles ( 30.0%)
3. colluvial_accumulations  :  43 tiles ( 43.0%)
4. residuum                 :  95 tiles ( 95.0%)

Total tiles: 100
Average labels per tile: 2.86

✓ All labels have at least 5 examples


In [11]:
# Configure Random Forest classifier
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print(f"{'='*60}")
print("5-FOLD CV RESULTS (Random splits)")
print(f"{'='*60}")
print("Metric: F1-macro (multi-label compatible)")
print(f"{'='*60}\n")

# Method configurations: (key, display_name)
methods = [
    ('traditional', 'Traditional'),
    ('ecc50', 'ECC-50d'),
    ('ecc200', 'ECC-200d'),
    ('betti100', 'Betti-100'),
    ('betti200', 'Betti-200'),
    ('landscape', 'Landscapes'),
    ('persimage', 'PersImages'),
    ('ph_directional', 'PH-Directional'),
]

# Compute cross-validation scores for all methods
cv_scores = {}
for key, display_name in methods:
    X = features[key]
    scores = cross_val_score(rf, X, y, cv=5, scoring='f1_macro')
    cv_scores[key] = scores
    
    print(f"{display_name} ({X.shape[1]}d):")
    print(f"  F1: {scores.mean():.3f} ± {scores.std():.3f}")
    print(f"  Per fold: {scores}\n")

# Summary comparison
print(f"{'='*60}")
print("SUMMARY COMPARISON")
print(f"{'='*60}\n")

# Create sorted results list
results = [(display_name, cv_scores[key], features[key].shape[1]) 
           for key, display_name in methods]
results_sorted = sorted(results, key=lambda x: x[1].mean(), reverse=True)

print("Ranked by F1-macro:")
for i, (name, scores, dims) in enumerate(results_sorted, 1):
    print(f"{i}. {name:15s} ({dims:4d}d): {scores.mean():.3f} ± {scores.std():.3f}")

# Timing summary
print(f"\n{'='*60}")
print("COMPUTATIONAL COST (per tile)")
print(f"{'='*60}")

# Individual method timings
print("\nIndividual method timings:")
timing_keys = {
    'traditional': 'traditional',
    'ecc50': 'ecc50',
    'ecc200': 'ecc200',
    'betti100': 'betti100',
    'betti200': 'betti200',
    'landscape': 'landscape',
    'persimage': 'persimage',
    'ph_directional': 'ph_directional'
}

for display_key, timing_key in timing_keys.items():
    if timing_key in timing:
        times = timing[timing_key]
        print(f"  {display_key:15s}: {np.mean(times):6.3f}s ± {np.std(times):5.3f}s")

# TDA pipeline breakdown
if 'ph_compute' in timing:
    print("\nTDA Pipeline Breakdown (PH computed once, reused for all vectorizations):")
    tda_components = [
        ('PH Computation', 'ph_compute'),
        ('Betti-100 vec', 'betti100'),
        ('Betti-200 vec', 'betti200'),
        ('Landscape vec', 'landscape'),
        ('PersImage vec', 'persimage'),
    ]
    
    for label, key in tda_components:
        print(f"  {label:16s}: {np.mean(timing[key]):6.3f}s")
    
    total_tda = sum(np.mean(timing[key]) for _, key in tda_components)
    print(f"  {'Total TDA':16s}: {total_tda:6.3f}s")

# Time comparison vs traditional
trad_time = np.mean(timing['traditional'])
print(f"\nTime ratios (vs. Traditional baseline at {trad_time:.3f}s):")

time_comparison = [
    ('ECC-50d', 'ecc50'),
    ('ECC-200d', 'ecc200'),
    ('Landscapes', 'landscape'),
    ('PersImages', 'persimage'),
    ('PH-Directional', 'ph_directional'),
]

for label, key in time_comparison:
    if key in timing:
        ratio = np.mean(timing[key]) / trad_time
        print(f"  {label:15s}: {ratio:5.2f}×")

if 'ph_compute' in timing:
    total_tda_time = sum(np.mean(timing[key]) for _, key in tda_components)
    ratio = total_tda_time / trad_time
    print(f"  {'Total TDA':15s}: {ratio:5.2f}×")

5-FOLD CV RESULTS (Random splits)
Metric: F1-macro (multi-label compatible)



d:\dev\karst-terrain-classification\.conda\Lib\site-packages\sklearn\metrics\_classification.py:1760: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))


Traditional (150d):
  F1: 0.668 ± 0.146
  Per fold: [0.58534799 0.70141231 0.69682589 0.89840367 0.45651106]

ECC-50d (50d):
  F1: 0.590 ± 0.110
  Per fold: [0.39487179 0.57740299 0.63349429 0.73255057 0.60918061]

ECC-200d (200d):
  F1: 0.563 ± 0.095
  Per fold: [0.39487179 0.57518926 0.60243726 0.68537109 0.55878378]

Betti-100 (200d):
  F1: 0.612 ± 0.095
  Per fold: [0.48058608 0.55902764 0.66445268 0.7601136  0.59425997]

Betti-200 (400d):
  F1: 0.613 ± 0.109
  Per fold: [0.48717949 0.55296703 0.66682762 0.79769602 0.55878378]

Landscapes (1100d):
  F1: 0.611 ± 0.081
  Per fold: [0.50915751 0.6196337  0.6006336  0.75483887 0.5730695 ]

PersImages (5000d):
  F1: 0.692 ± 0.102
  Per fold: [0.59487179 0.73487179 0.68605003 0.86277538 0.58378378]

PH-Directional (800d):
  F1: 0.541 ± 0.089
  Per fold: [0.406934   0.51261332 0.57449839 0.67973616 0.52924584]

SUMMARY COMPARISON

Ranked by F1-macro:
1. PersImages      (5000d): 0.692 ± 0.102
2. Traditional     ( 150d): 0.668 ± 0.146
3. Be

In [12]:
# Save CV results for use in summary_report.ipynb
cv_results = {
    'scores': cv_scores,
    'shapes': {key: features[key].shape[1] for key in cv_scores.keys()}
}

cv_results_path = npy_folder_path / 'testbed_cv_results.npy'
np.save(cv_results_path, cv_results, allow_pickle=True)

print(f"\n✓ CV results saved to {cv_results_path}")
print("  This allows summary_report.ipynb to reuse these results without recomputing")


✓ CV results saved to D:\dev\karst-terrain-classification\data\processed\minimal_exploration\testbed_cv_results.npy
  This allows summary_report.ipynb to reuse these results without recomputing
